In [ ]:
# Standard Library
import os
import random
import glob

# Data Manipulation & Storage
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from tqdm.notebook import tqdm, trange

# Machine Learning & Statistics
import joblib
import optuna
from optuna_dashboard import save_plotly_graph_object
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

# PyTorch Core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset

# PyTorch Geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, HeteroConv
from torch_geometric.loader import DataLoader

# Global Configuration
sns.set(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Determinism / Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# --- GLOBAL CONSTANTS ---
# The primary root for all data
BASE_DIR = '../Data/Models/Model_1' 

# Directory paths
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
PROCESSED_TRAIN_DIR = os.path.join(TRAIN_DIR, 'Processed')
PROCESSED_VAL_DIR = os.path.join(BASE_DIR, 'val', 'Processed')
PROCESSED_TEST_DIR = os.path.join(BASE_DIR, 'internal_test', 'Processed')

# Static Node Files
DF_1D_STATIC_PATH = os.path.join(TRAIN_DIR, '1d_nodes_static.csv')
DF_2D_STATIC_PATH = os.path.join(TRAIN_DIR, '2d_nodes_static.csv')

# Edge & Connection Files
PATH_1D_EDGES = os.path.join(TRAIN_DIR, '1d_edge_index.csv')
PATH_2D_EDGES = os.path.join(TRAIN_DIR, '2d_edge_index.csv')
PATH_CONNECTIONS = os.path.join(TRAIN_DIR, '1d2d_connections.csv')

# Search Patterns (Optional, for globbing)
TRAIN_PATTERN = os.path.join(PROCESSED_TRAIN_DIR, '*.pt')
VAL_PATTERN   = os.path.join(PROCESSED_VAL_DIR, '*.pt')
TEST_PATTERN  = os.path.join(PROCESSED_TEST_DIR, '*.pt')

# Join the directory and the search pattern in one go
TRAIN_PATTERN = os.path.join(BASE_DIR, 'train', 'Processed', '*.pt')
VAL_PATTERN   = os.path.join(BASE_DIR, 'val', 'Processed', '*.pt')
TEST_PATTERN  = os.path.join(BASE_DIR, 'internal_test', 'Processed', '*.pt')

# Model Save Paths
model_1_save_path = 'best_flood_model_1.pth'
model_2_save_path = 'best_flood_model_2.pth'

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, file_paths):
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the pre-processed HeteroData object from disk
        return torch.load(self.file_paths[idx])

In [ ]:

# 1. Collect all processed .pt file paths for each split
# Using glob to grab every file in the Processed directories we just created
train_files = glob.glob(TRAIN_PATTERN)
val_files = glob.glob(VAL_PATTERN)
test_files = glob.glob(TEST_PATTERN)

# 2. Initialize the Datasets
# Assuming your FloodDataset class is designed to take a list of file paths
train_ds = FloodDataset(train_files)
val_ds = FloodDataset(val_files)
test_ds = FloodDataset(test_files)

# 3. Create the Loaders
# shuffle=True for training to ensure the model doesn't learn the order of storms
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# Quick sanity check for your report
print("Loaders initialized:")
print(f" - Training Batches: {len(train_loader)}")
print(f" - Validation Batches: {len(val_loader)}")
print(f" - Test Batches: {len(test_loader)}")

### Baseline Model, Random Forest

In [ ]:
def prepare_tabular_data(loader):
    """
    Flattens HeteroData objects into X (features) and y (targets).
    For the baseline, we predict the 'next' timestep for each node.
    """
    X_list, y_list = [], []
    
    for data in loader:
        # data['node1d'].x shape is [Timesteps, Nodes, Features]
        # We'll focus on 1D nodes for this example baseline
        x_1d = data['node1d'].x.numpy()
        
        # Create sliding window: Use time 't' to predict 't+1'
        # Features: [t], Target: [t+1, water_depth_index]
        # Assuming water depth is the last dynamic feature added
        X_frames = x_1d[:-1, :, :] # All steps except last
        y_frames = x_1d[1:, :, -1:] # All steps except first, depth only
        
        # Flatten [Timesteps * Nodes, Features]
        X_flat = X_frames.reshape(-1, X_frames.shape[-1])
        y_flat = y_frames.reshape(-1)
        
        X_list.append(X_flat)
        y_list.append(y_flat)
        
    return np.vstack(X_list), np.concatenate(y_list)

In [ ]:
print("Flattening data for Random Forest...")
X_train, y_train = prepare_tabular_data(train_loader)
X_val, y_val = prepare_tabular_data(train_loader)  # Using train_loader for validation as well, since val_loader is empty

rf_baseline = RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)

print("Training Random Forest Baseline...")
rf_baseline.fit(X_train, y_train)

# 3. Evaluate
y_pred = rf_baseline.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f"Baseline Validation RMSE: {rmse:.4f}")

In [ ]:
# Configuration
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

COLORS = {
    'input': '#1a252f',      
    'baseline': '#566573',   
    'primary': '#1f618d',    
    'output': '#a04000',     
    'bg': '#ffffff',         
    'text_light': '#ffffff',
    'text_dark': '#1a252f'
}

# Adjusted width/height for maximum efficiency
fig, ax = plt.subplots(figsize=(16, 4.5), facecolor=COLORS['bg'])
# Tightened xlim (0 to 14 instead of 16) to remove side gaps
ax.set_xlim(0, 14)
ax.set_ylim(0, 4.5) 
ax.axis('off')

def draw_compact_box(x, y, w, h, title, subtitle, color):
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.05", 
                          linewidth=0, facecolor=color, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.62, title, ha='center', va='center', 
            fontsize=13, fontweight='black', color=COLORS['text_light'], zorder=3)
    ax.text(x + w/2, y + h*0.32, subtitle, ha='center', va='center', 
            fontsize=10, fontweight='bold', color=COLORS['text_light'], zorder=3)

# Title - Positioned to minimize top whitespace
ax.text(7, 4.1, "SYSTEM ARCHITECTURE: TOPOLOGICAL VS. STATISTICAL MODELING", 
        fontsize=18, fontweight='black', ha='center', color=COLORS['text_dark'])

# 1. Shared Input (Left)
draw_compact_box(0.2, 1.2, 2.8, 1.6, "1. SOURCE DATA", 
                "Heterogeneous Graph\nNodes, Edges, Features", COLORS['input'])

# 2. Baseline Pathway (Middle Top)
draw_compact_box(5.0, 2.3, 4.0, 1.3, "BASELINE BENCHMARK", 
                "Random Forest (Topology-Blind)", COLORS['baseline'])

# 3. Proposed Pipeline (Middle Bottom)
draw_compact_box(5.0, 0.4, 4.0, 1.3, "PROPOSED: FloodGNN", 
                "ST-GNN (Topology-Aware)", COLORS['primary'])

# 4. Evaluation/Output (Right)
draw_compact_box(11.0, 1.2, 2.8, 1.6, "2. EVALUATION", 
                "Water Level Predictions\nRMSE / $R^2$ Score", COLORS['output'])

# Thicker arrows
arrow_props = dict(arrowstyle='-|> ,head_width=0.4,head_length=0.6', color='#34495e', lw=2.5)

# Connectors
ax.annotate('', xy=(5.0, 2.95), xytext=(3.0, 2.0), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=90,rad=5", **arrow_props))
ax.annotate('', xy=(5.0, 1.05), xytext=(3.0, 2.0), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=-90,rad=5", **arrow_props))
ax.annotate('', xy=(11.0, 2.0), xytext=(9.0, 2.95), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=-90,rad=5", **arrow_props))
ax.annotate('', xy=(11.0, 2.0), xytext=(9.0, 1.05), arrowprops=dict(connectionstyle="angle,angleA=0,angleB=90,rad=5", **arrow_props))

# Labels
ax.text(4.0, 3.1, "Data Flattening", fontsize=10, fontweight='bold', ha='center', style='italic')
ax.text(4.0, 0.7, "Graph Maintained", fontsize=10, fontweight='bold', ha='center', style='italic')

# Final Save with absolute minimal padding
plt.savefig('architecture_no_whitespace.png', dpi=300, bbox_inches='tight', pad_inches=0.05)
plt.show()

### Primary Model

In [ ]:
class FloodGNN(nn.Module):
    def __init__(self, hidden_channels, out_channels, dropout_p=0.2):
        super().__init__()
        self.dropout_p = dropout_p
        
        # 1. Spatial Layers
        self.conv1 = HeteroConv({
            ('node1d', 'pipe', 'node1d'): SAGEConv((-1, -1), hidden_channels),
            ('node2d', 'surface', 'node2d'): SAGEConv((-1, -1), hidden_channels),
            ('node1d', 'exchange', 'node2d'): SAGEConv((-1, -1), hidden_channels),
            ('node2d', 'exchange', 'node1d'): SAGEConv((-1, -1), hidden_channels),
        }, aggr='sum')

        self.conv2 = HeteroConv({
            ('node1d', 'pipe', 'node1d'): SAGEConv(hidden_channels, hidden_channels),
            ('node2d', 'surface', 'node2d'): SAGEConv(hidden_channels, hidden_channels),
            ('node1d', 'exchange', 'node2d'): SAGEConv(hidden_channels, hidden_channels),
            ('node2d', 'exchange', 'node1d'): SAGEConv(hidden_channels, hidden_channels),
        }, aggr='sum')

        # 2. Temporal Layers
        self.gru_1d = nn.GRU(hidden_channels, hidden_channels, batch_first=True)
        self.gru_2d = nn.GRU(hidden_channels, hidden_channels, batch_first=True)

        # 3. Output Heads
        self.dropout = nn.Dropout(p=dropout_p)
        self.lin_1d = nn.Linear(hidden_channels, out_channels)
        self.lin_2d = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, h_dict=None):
        # --- Layer 1 ---
        h_dict_spatial = self.conv1(x_dict, edge_index_dict)
        h_dict_spatial = {key: torch.relu(h) for key, h in h_dict_spatial.items()}
        h_dict_spatial = {key: self.dropout(h) for key, h in h_dict_spatial.items()}

        # --- Layer 2 ---
        h_dict_spatial = self.conv2(h_dict_spatial, edge_index_dict)
        h_dict_spatial = {key: torch.relu(h) for key, h in h_dict_spatial.items()}
        h_dict_spatial = {key: self.dropout(h) for key, h in h_dict_spatial.items()}
        
        # --- Temporal Pass ---
        # node1d
        x_1d = h_dict_spatial['node1d'].unsqueeze(1)
        h_1d, next_h1d = self.gru_1d(x_1d, h_dict['node1d'] if h_dict else None)
        
        # node2d
        x_2d = h_dict_spatial['node2d'].unsqueeze(1)
        h_2d, next_h2d = self.gru_2d(x_2d, h_dict['node2d'] if h_dict else None)

        # Apply dropout one last time before the final linear layer
        out_1d = self.lin_1d(self.dropout(h_1d.squeeze(1)))
        out_2d = self.lin_2d(self.dropout(h_2d.squeeze(1)))
        
        return out_1d, out_2d, {'node1d': next_h1d, 'node2d': next_h2d}

In [ ]:
def calculate_dataset_std(loader):
    all_1d_levels = []
    all_2d_levels = []
    
    print("Calculating dataset statistics...")
    for batch in tqdm(loader):
        # Extract the target water level (last column of the feature matrix)
        # Shape of x: [Timesteps, Nodes, Features]
        y_1d = batch['node1d'].x[:, :, -1].flatten()
        y_2d = batch['node2d'].x[:, :, -1].flatten()
        
        all_1d_levels.append(y_1d.numpy())
        all_2d_levels.append(y_2d.numpy())
    
    # Concatenate all events into one giant array
    full_1d = np.concatenate(all_1d_levels)
    full_2d = np.concatenate(all_2d_levels)
    
    # Calculate Standard Deviation
    std_1d = np.std(full_1d)
    std_2d = np.std(full_2d)
    
    return std_1d, std_2d

In [ ]:
std_1d, std_2d = calculate_dataset_std(train_loader)

# Create your own dictionary for the loss function
std_dev_dict = {
    (1, 1): std_1d, 
    (1, 2): std_2d,
    (2, 1): std_1d, # If Model 2 is trained on the same data, use the same std
    (2, 2): std_2d
}

print(f"\nCalculated 1D Std: {std_1d:.4f}")
print(f"Calculated 2D Std: {std_2d:.4f}")

In [ ]:
class StandardizedRMSELoss(nn.Module):
    def __init__(self, std_dev_dict):
        super().__init__()
        self.std_dev_dict = std_dev_dict
    
    def forward(self, pred_1d, pred_2d, target_1d, target_2d, model_id):
        # Get std devs for this model
        std_1d = self.std_dev_dict[(model_id, 1)]
        std_2d = self.std_dev_dict[(model_id, 2)]
        
        # Calculate RMSE for each node type
        rmse_1d = torch.sqrt(torch.mean((pred_1d - target_1d) ** 2) + 1e-6)
        rmse_2d = torch.sqrt(torch.mean((pred_2d - target_2d) ** 2) + 1e-6)
        
        # Standardize by dividing by std dev
        std_rmse_1d = rmse_1d / std_1d
        std_rmse_2d = rmse_2d / std_2d
        
        # Average across node types
        return (std_rmse_1d + std_rmse_2d) / 2

In [ ]:
def validate_event_with_metrics(model, event_data, criterion, model_id):
    model.eval()
    h_dict = None
    num_t = event_data['node1d'].x.size(0)
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]
    
    total_loss = 0
    all_preds = []
    all_targets = []

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        with torch.no_grad():
            pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        total_loss += criterion(pred_1d, pred_2d, target_1d, target_2d, model_id).item()

        # Collect for residuals (using 1D as the primary metric for example)
        all_preds.append(pred_1d.cpu().numpy().flatten())
        all_targets.append(target_1d.cpu().numpy().flatten())

        # Autoregressive step
        next_x_1d = event_data['node1d'].x[t].clone()
        next_x_1d[:, -1] = pred_1d.squeeze()
        next_x_2d = event_data['node2d'].x[t].clone()
        next_x_2d[:, -1] = pred_2d.squeeze()
        current_x_1d, current_x_2d = next_x_1d, next_x_2d

    return total_loss / (num_t - 1), np.concatenate(all_preds), np.concatenate(all_targets)

def train_event(model, event_data, optimizer, criterion, model_id, 
                teacher_forcing_ratio=0.5, max_grad_norm=1.0):
    model.train()
    event_data = event_data.to(device)
    optimizer.zero_grad()
    
    num_t = event_data['node1d'].x.size(0)
    accumulated_loss = 0 
    running_loss_val = 0.0
    
    h_dict = None 
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        loss = criterion(pred_1d, pred_2d, target_1d, target_2d, model_id)
        accumulated_loss = accumulated_loss + loss
        running_loss_val += loss.item()
        
        # Teacher Forcing Logic
        if random.random() < teacher_forcing_ratio:
            current_x_1d = event_data['node1d'].x[t]
            current_x_2d = event_data['node2d'].x[t]
        else:
            # Autoregressive: feedback the prediction into the next time step
            next_x_1d = event_data['node1d'].x[t].clone()
            next_x_1d[:, -1] = pred_1d.detach().squeeze()
            next_x_2d = event_data['node2d'].x[t].clone()
            next_x_2d[:, -1] = pred_2d.detach().squeeze()
            current_x_1d, current_x_2d = next_x_1d, next_x_2d

    # Optimization
    accumulated_loss.backward()
    # Tunable gradient clipping to prevent exploding gradients in recurrent loops
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()
    
    return running_loss_val / (num_t - 1)


def train_model(model, train_loader, val_loader, optimizer, criterion, model_id, 
                num_epochs=50, initial_tf=0.5, tf_decay=0.05, min_tf=0.1,
                patience=5, min_delta=0.001):
    
    train_losses, val_losses = [], []
    r2_history = []
    best_val_loss = float('inf')
    epochs_no_improve = 0
    SAVE_PATH = f'best_model_{model_id}.pth'
    
    val_residuals = None 

    for epoch in range(num_epochs):
        current_tf = max(min_tf, initial_tf - (epoch * tf_decay))
        
        # TRAINING PHASE
        model.train()
        total_train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            total_train_loss += train_event(model, batch, optimizer, criterion, model_id, current_tf)
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # VALIDATION PHASE
        model.eval() # Added explicit eval mode
        epoch_preds, epoch_targets = [], []
        total_val_loss = 0
        
        with torch.no_grad(): # Ensure no gradients are tracked
            for batch in val_loader:
                v_loss, p, t = validate_event_with_metrics(model, batch.to(device), criterion, model_id)
                total_val_loss += v_loss
                epoch_preds.append(p)
                epoch_targets.append(t)
        
        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        full_preds = np.concatenate(epoch_preds)
        full_targets = np.concatenate(epoch_targets)
        current_r2 = r2_score(full_targets, full_preds)
        r2_history.append(current_r2)

        # EARLY STOPPING & CHECKPOINTING LOGIC
        if avg_val_loss < (best_val_loss - min_delta):
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), SAVE_PATH)
            val_residuals = full_targets - full_preds
            epochs_no_improve = 0
            print(f" >>> Checkpoint saved: Val Loss {avg_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f" !!! Early stopping triggered at epoch {epoch+1}")
                break 

        print(f"E{epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | R2: {current_r2:.4f}")

    return train_losses, val_losses, {
        'r2': r2_history,
        'residuals': val_residuals,
        'best_loss': best_val_loss
    }

def validate_event(model, event_data, criterion, model_id):
    """Simplified version of train_event without backprop or teacher forcing"""
    h_dict = None
    num_t = event_data['node1d'].x.size(0)
    current_x_1d = event_data['node1d'].x[0]
    current_x_2d = event_data['node2d'].x[0]
    total_loss = 0

    for t in range(1, num_t):
        x_dict = {'node1d': current_x_1d, 'node2d': current_x_2d}
        pred_1d, pred_2d, h_dict = model(x_dict, event_data.edge_index_dict, h_dict)
        
        target_1d = event_data['node1d'].x[t, :, -1].unsqueeze(-1)
        target_2d = event_data['node2d'].x[t, :, -1].unsqueeze(-1)
        
        total_loss += criterion(pred_1d, pred_2d, target_1d, target_2d, model_id).item()

        # Autoregressive step: always use predictions for validation
        next_x_1d = event_data['node1d'].x[t].clone()
        next_x_1d[:, -1] = pred_1d.squeeze()
        next_x_2d = event_data['node2d'].x[t].clone()
        next_x_2d[:, -1] = pred_2d.squeeze()
        current_x_1d, current_x_2d = next_x_1d, next_x_2d

    return total_loss / (num_t - 1)

def finalize_test_metrics(model, test_loader, criterion, model_id, save_path):
    # Load the best state found during validation
    model.load_state_dict(torch.load(save_path))
    model.eval()
    
    test_preds, test_targets = [], []
    total_test_loss = 0
    
    for batch in test_loader:
        loss, p, t = validate_event_with_metrics(model, batch.to(device), criterion, model_id)
        total_test_loss += loss
        test_preds.append(p)
        test_targets.append(t)
        
    avg_test_loss = total_test_loss / len(test_loader)
    all_p = np.concatenate(test_preds)
    all_t = np.concatenate(test_targets)
    test_r2 = r2_score(all_t, all_p)
    
    print(f"\n--- FINAL TEST RESULTS ---")
    print(f"Test Loss: {avg_test_loss:.4f}")
    print(f"Test R2: {test_r2:.4f}")
    
    return avg_test_loss, test_r2, (all_t - all_p)

In [ ]:
class WeightedStandardizedRMSELoss(nn.Module):
    def __init__(self, std_dev_dict, weight_1d=0.7):
        super().__init__()
        self.std_dev_dict = std_dev_dict
        self.weight_1d = weight_1d
        self.weight_2d = 1.0 - weight_1d

    def forward(self, pred_1d, pred_2d, target_1d, target_2d, model_id):
        # Get std devs for this model
        std_1d = self.std_dev_dict[(model_id, 1)]
        std_2d = self.std_dev_dict[(model_id, 2)]
        
        # Calculate RMSE for each node type
        rmse_1d = torch.sqrt(torch.mean((pred_1d - target_1d) ** 2) + 1e-6)
        rmse_2d = torch.sqrt(torch.mean((pred_2d - target_2d) ** 2) + 1e-6)
        
        # Standardize
        std_rmse_1d = rmse_1d / std_1d
        std_rmse_2d = rmse_2d / std_2d
        
        # Weighted average to prioritize sewer (1D) performance
        return (self.weight_1d * std_rmse_1d) + (self.weight_2d * std_rmse_2d)

In [ ]:
def objective(trial):
    #Hyperparameter Search Space
    hidden_channels = trial.suggest_categorical("hidden_channels", [8, 16, 32])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-4, log=True)
    dropout_p = trial.suggest_float("dropout_p", 0.1, 0.4)
    initial_tf = trial.suggest_float("initial_tf", 0.4, 0.6)
    tf_decay = trial.suggest_float("tf_decay", 0.02, 0.05)

    # Model & Optimizer Setup
    model = FloodGNN(
        hidden_channels=hidden_channels, 
        out_channels=1, 
        dropout_p=dropout_p
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = WeightedStandardizedRMSELoss(std_dev_dict, weight_1d=0.7)
    
    # Lean Training Loop 
    # We only look at a small subset of the data to find the best PARAMS
    # Once found, train on the FULL data outside of Optuna
    TRAIN_BATCH_LIMIT = 40  
    VAL_BATCH_LIMIT = 20    
    NUM_EPOCHS = 30

    best_val_loss = float('inf')

    for epoch in range(NUM_EPOCHS):
        current_tf = max(0.2, initial_tf - (epoch * tf_decay))
        
        # SUBSET TRAINING
        model.train()
        for i, batch in enumerate(train_loader):
            if i >= TRAIN_BATCH_LIMIT: break 
            _ = train_event(model, batch, optimizer, criterion, model_id=1, teacher_forcing_ratio=current_tf)
        
        # SUBSET VALIDATION
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for i, batch in enumerate(val_loader):
                if i >= VAL_BATCH_LIMIT: break
                v_loss = validate_event(model, batch.to(device), criterion, model_id=1)
                total_val_loss += v_loss
        
        avg_val_loss = total_val_loss / VAL_BATCH_LIMIT

        # Optuna Reporting & Pruning 
        trial.report(avg_val_loss, epoch)
        
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            
    return best_val_loss

In [ ]:
# Using a local SQLite DB so you don't lose progress if it crashes
study = optuna.create_study(
    study_name="flood_gnn_fast_tune",
    storage="sqlite:///db.sqlite3",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(min_resource=2), # Don't prune until epoch 2
    load_if_exists=True
)

print("Starting Fast Optimization...")
study.optimize(objective, n_trials=15, show_progress_bar=True)

print("\nOptimization Complete!")
print("Best Hyperparams:", study.best_params)

In [ ]:
# --- 1. Setup the Winning Architecture ---
# Note: Use the exact values from study.best_params
best_params = study.best_params

model_final = FloodGNN(
    hidden_channels=best_params['hidden_channels'], 
    out_channels=1, 
    dropout_p=best_params['dropout_p']
).to(device)

# --- 2. Final Optimizer & Loss ---
optimizer_final = torch.optim.Adam(
    model_final.parameters(), 
    lr=best_params['lr'], 
    weight_decay=best_params['weight_decay']
)

# Reuse your standardized criterion
criterion = WeightedStandardizedRMSELoss(std_dev_dict, weight_1d=0.7)

# --- 3. Final Training Run (The Marathon) ---
# Use the full train_loader and val_loader here, not the subsets!
history_final = train_model(
    model=model_final, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    optimizer=optimizer_final, 
    criterion=criterion, 
    model_id=1, # <--- Change "tuned_final" back to 1
    num_epochs=40,
    initial_tf=best_params['initial_tf'], 
    tf_decay=best_params['tf_decay'], 
    min_tf=0.2
)

In [ ]:
def evaluate_model_comprehensive(model, loader, device, model_name="Model", save_path=None):
    """
    Consolidates evaluation: Loads weights, runs autoregressive inference, 
    calculates metrics, and returns a dictionary ready for scaling/plotting.
    """
    if save_path:
        model.load_state_dict(torch.load(save_path))
        print(f"Loaded weights from {save_path}")
        
    model.eval()
    results = {
        '1d': {'preds': [], 'actuals': []},
        '2d': {'preds': [], 'actuals': []}
    }

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluating {model_name}"):
            batch = batch.to(device)
            # Assuming batch has a time dimension at index 0
            num_t = batch['node1d'].x.size(0)
            h_dict = None
            
            # Initial state (t=0)
            curr_x1d, curr_x2d = batch['node1d'].x[0], batch['node2d'].x[0]

            for t in range(1, num_t):
                x_dict = {'node1d': curr_x1d, 'node2d': curr_x2d}
                p1d, p2d, h_dict = model(x_dict, batch.edge_index_dict, h_dict)
                
                # Store results for metrics
                results['1d']['preds'].append(p1d.cpu().numpy())
                results['1d']['actuals'].append(batch['node1d'].x[t, :, -1].cpu().numpy())
                
                results['2d']['preds'].append(p2d.cpu().numpy())
                results['2d']['actuals'].append(batch['node2d'].x[t, :, -1].cpu().numpy())
                
                # Autoregressive Update: Prediction becomes input for t+1
                next_x1d = batch['node1d'].x[t].clone()
                next_x1d[:, -1] = p1d.squeeze()
                next_x2d = batch['node2d'].x[t].clone()
                next_x2d[:, -1] = p2d.squeeze()
                curr_x1d, curr_x2d = next_x1d, next_x2d

    # Flatten for global metrics
    p1d_all = np.concatenate([p.flatten() for p in results['1d']['preds']])
    a1d_all = np.concatenate([a.flatten() for a in results['1d']['actuals']])
    p2d_all = np.concatenate([p.flatten() for p in results['2d']['preds']])
    a2d_all = np.concatenate([a.flatten() for a in results['2d']['actuals']])
    
    rmse_1d = np.sqrt(np.mean((p1d_all - a1d_all)**2))
    rmse_2d = np.sqrt(np.mean((p2d_all - a2d_all)**2))
    combined_r2 = r2_score(np.concatenate([a1d_all, a2d_all]), np.concatenate([p1d_all, p2d_all]))
    
    print(f"\n--- {model_name} Results ---")
    print(f"1D RMSE: {rmse_1d:.4f} | 2D RMSE: {rmse_2d:.4f} | R2: {combined_r2:.4f}")

    # This dictionary is the "source of truth" for the next steps
    eval_metrics = {
        'preds_1d': results['1d']['preds'],
        'preds_2d': results['2d']['preds'],
        'residuals': (a1d_all - p1d_all),
        'r2_final': combined_r2,
        'rmse_1d': rmse_1d,
        'rmse_2d': rmse_2d,
        'model_name': model_name
    }
    return eval_metrics

def get_original_scale_predictions(eval_metrics, scaler_1d_path, scaler_2d_path):
    """
    Inverse transforms standardized predictions back to physical units.
    """
    s1d = joblib.load(scaler_1d_path)
    s2d = joblib.load(scaler_2d_path)

    # Water level is at the last index
    idx = -1
    scale1, mean1 = s1d.scale_[idx], s1d.mean_[idx]
    scale2, mean2 = s2d.scale_[idx], s2d.mean_[idx]

    # Apply x = (z * sigma) + mu and clip physically impossible negatives
    preds_1d_phys = [np.clip(p * scale1 + mean1, 0, None) for p in eval_metrics['preds_1d']]
    preds_2d_phys = [np.clip(p * scale2 + mean2, 0, None) for p in eval_metrics['preds_2d']]

    print(f"--- {eval_metrics['model_name']} Physical Scale ---")
    print(f"1D Range: {np.min([p.min() for p in preds_1d_phys]):.3f}m to {np.max([p.max() for p in preds_1d_phys]):.3f}m")
    
    return {'1d_m': preds_1d_phys, '2d_m': preds_2d_phys}

def plot_learning_curve(history_tuple, model_name):
    """
    history_tuple: (train_losses, val_losses, eval_metrics)
    """
    train_losses, val_losses, eval_metrics = history_tuple
    epochs = range(1, len(train_losses) + 1)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

    ax1.plot(epochs, train_losses, label='Train')
    ax1.plot(epochs, val_losses, label='Val', linestyle='--')
    ax1.set_title(f'{model_name}: Loss')
    ax1.legend()

    gap = np.array(val_losses) / np.array(train_losses)
    ax2.plot(epochs, gap, color='red')
    ax2.axhline(y=1, color='black', linestyle=':')
    ax2.set_title('Overfitting Ratio (Val/Train)')

    if 'residuals' in eval_metrics:
        ax3.hist(eval_metrics['residuals'], bins=50, color='purple', alpha=0.7)
        ax3.axvline(x=0, color='black')
        ax3.set_title('Residual Distribution (Std units)')

    plt.tight_layout()
    plt.show()

In [ ]:
eval_for_plot = eval_results if 'eval_results' in globals() else history_final[2]

plot_learning_curve(
    (history_final[0], history_final[1], eval_for_plot),
    "Tuned_FloodGNN_vFinal"
)

In [ ]:
# Load trained FloodGNN weights from file
weight_path = 'best_model_1.pth'

if 'model_final' in globals():
    model_final.load_state_dict(torch.load(weight_path, map_location=device))
else:
    best_params = study.best_params
    model_final = FloodGNN(
        hidden_channels=best_params['hidden_channels'],
        out_channels=1,
        dropout_p=best_params['dropout_p']
    ).to(device)
    model_final.load_state_dict(torch.load(weight_path, map_location=device))

model_final.eval()
print(f"Model loaded from: {weight_path}")

In [ ]:
# Run inference and get metrics
eval_results = evaluate_model_comprehensive(
    model_final, 
    test_loader, 
    device, 
    model_name="Tuned_FloodGNN_vFinal", 
    save_path='best_model_1.pth'
)

# Plot the definitive learning curve
# history_final should be the (train_losses, val_losses, tf_history) returned by train_model
plot_learning_curve(
    (history_final[0], history_final[1], eval_results), 
    "Tuned_FloodGNN_vFinal"
)

# Final Scale-Back to Physical Units (Meters)
physical_preds = get_original_scale_predictions(
    eval_results, 'scaler_dyn_1d.pkl', 'scaler_dyn_2d.pkl'
)

In [ ]:
# One-step inference diagnostics (t=0 -> t=1)
model = model_final
model.load_state_dict(torch.load('best_model_1.pth'))  # optional but recommended
model.eval()

event_data = next(iter(test_loader)).to(device)
h_dict = None

x_dict = {
    'node1d': event_data['node1d'].x[0],  # input at t=0
    'node2d': event_data['node2d'].x[0]
}

with torch.no_grad():
    pred_1d, pred_2d, h_dict_next = model(x_dict, event_data.edge_index_dict, h_dict)

# Ground truth at next timestep (t=1)
target_1d = event_data['node1d'].x[1, :, -1].unsqueeze(-1)
target_2d = event_data['node2d'].x[1, :, -1].unsqueeze(-1)

# Errors
err_1d = (pred_1d - target_1d).squeeze()
err_2d = (pred_2d - target_2d).squeeze()

rmse_1d = torch.sqrt(torch.mean(err_1d**2)).item()
rmse_2d = torch.sqrt(torch.mean(err_2d**2)).item()

r2_1d = r2_score(target_1d.detach().cpu().numpy(), pred_1d.detach().cpu().numpy())
r2_2d = r2_score(target_2d.detach().cpu().numpy(), pred_2d.detach().cpu().numpy())

print("Shapes:", pred_1d.shape, pred_2d.shape)
print(f"1D one-step RMSE: {rmse_1d:.4f} | R2: {r2_1d:.4f}")
print(f"2D one-step RMSE: {rmse_2d:.4f} | R2: {r2_2d:.4f}")

print("\n1D prediction stats:")
print(f"  pred mean={pred_1d.mean().item():.4f}, min={pred_1d.min().item():.4f}, max={pred_1d.max().item():.4f}")
print(f"  true mean={target_1d.mean().item():.4f}, min={target_1d.min().item():.4f}, max={target_1d.max().item():.4f}")

print("\n2D prediction stats:")
print(f"  pred mean={pred_2d.mean().item():.4f}, min={pred_2d.min().item():.4f}, max={pred_2d.max().item():.4f}")
print(f"  true mean={target_2d.mean().item():.4f}, min={target_2d.min().item():.4f}, max={target_2d.max().item():.4f}")

# Show worst 5 node errors (1D)
abs_err_1d = torch.abs(err_1d)
topk_vals, topk_idx = torch.topk(abs_err_1d, k=5)
print("\nWorst 1D nodes (index, abs error):")
for i in range(5):
    print(int(topk_idx[i]), float(topk_vals[i]))

# Quick visual check (1D)
plt.figure(figsize=(10, 4))
plt.plot(target_1d.detach().cpu().numpy().flatten(), label='True 1D (t=1)')
plt.plot(pred_1d.detach().cpu().numpy().flatten(), label='Pred 1D (t=1)', alpha=0.8)
plt.title("One-step prediction: 1D nodes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare baseline (Random Forest) vs FloodGNN on the SAME one-step 1D task

def evaluate_gnn_one_step_1d(model, loader, weight_path='best_model_1.pth'):
    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.eval()

    preds, targets = [], []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            h_dict = None
            num_t = batch['node1d'].x.size(0)

            # one-step teacher-forced evaluation: use true x[t-1] to predict y[t]
            for t in range(1, num_t):
                x_dict = {
                    'node1d': batch['node1d'].x[t - 1],
                    'node2d': batch['node2d'].x[t - 1]
                }
                pred_1d, _, h_dict = model(x_dict, batch.edge_index_dict, h_dict)
                target_1d = batch['node1d'].x[t, :, -1].unsqueeze(-1)

                preds.append(pred_1d.detach().cpu().numpy().ravel())
                targets.append(target_1d.detach().cpu().numpy().ravel())

    p = np.concatenate(preds)
    y = np.concatenate(targets)
    rmse = np.sqrt(mean_squared_error(y, p))
    r2 = r2_score(y, p)
    return rmse, r2

# Baseline on test set
X_test, y_test = prepare_tabular_data(test_loader)
baseline_pred = rf_baseline.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

# FloodGNN on test set (comparable one-step setup)
gnn_rmse, gnn_r2 = evaluate_gnn_one_step_1d(model_final, test_loader, weight_path='best_model_1.pth')

Verdict
print("=== One-step 1D Test Comparison ===")
print(f"Baseline RF     -> RMSE: {baseline_rmse:.4f}, R2: {baseline_r2:.4f}")
print(f"FloodGNN (best) -> RMSE: {gnn_rmse:.4f}, R2: {gnn_r2:.4f}")

outperforms = (gnn_rmse < baseline_rmse) and (gnn_r2 > baseline_r2)
print(f"\nOutperforming baseline? {'YES' if outperforms else 'NO'}")

In [ ]:
import optuna
from optuna_dashboard import save_plotly_graph_object

# Reuse existing study (already connected to storage)
storage = "sqlite:///db.sqlite3"
study = optuna.load_study(study_name="flood_gnn_fast_tune", storage=storage)

# 1) Final values chart (optimization history)
fig_final = optuna.visualization.plot_optimization_history(study)
fig_final.update_layout(title="Final Objective Values")
save_plotly_graph_object(study, fig_final)

# 2) Intermediate values chart
fig_intermediate = optuna.visualization.plot_intermediate_values(study)
fig_intermediate.update_layout(title="Intermediate Values")
save_plotly_graph_object(study, fig_intermediate)

# 3) Variable importance chart
fig_importance = optuna.visualization.plot_param_importances(study)
fig_importance.update_layout(title="Hyperparameter Importance")
save_plotly_graph_object(study, fig_importance)

# Optional: also save as standalone HTML files
fig_final.write_html("final_values.html")
fig_intermediate.write_html("intermediate_values.html")
fig_importance.write_html("variable_importance.html")

In [ ]:
# Reuse existing study (already connected to storage)
storage = "sqlite:///db.sqlite3"
study = optuna.load_study(study_name="flood_gnn_fast_tune", storage=storage)

# 1) Final values chart (optimization history)
fig_final = optuna.visualization.plot_optimization_history(study)
fig_final.update_layout(title="Final Objective Values")
save_plotly_graph_object(study, fig_final)

# 2) Intermediate values chart
fig_intermediate = optuna.visualization.plot_intermediate_values(study)
fig_intermediate.update_layout(title="Intermediate Values")
save_plotly_graph_object(study, fig_intermediate)

# 3) Variable importance chart
fig_importance = optuna.visualization.plot_param_importances(study)
fig_importance.update_layout(title="Hyperparameter Importance")
save_plotly_graph_object(study, fig_importance)

# Optional: also save as standalone HTML files
fig_final.write_html("final_values.html")
fig_intermediate.write_html("intermediate_values.html")
fig_importance.write_html("variable_importance.html")

In [ ]:
best_trial = study.best_trial

print(f"Best trial number: {best_trial.number}")
print(f"Best objective value: {best_trial.value:.6f}")
print("Best hyperparameters:")
for name, value in best_trial.params.items():
    print(f"  - {name}: {value}")

In [ ]:
true_candidates = ["y_true", "true_values", "targets", "y_test", "actual_values"]
pred_candidates = ["y_pred", "predictions", "y_hat", "pred_values", "y_test_pred"]

y_true = next((globals()[k] for k in true_candidates if k in globals()), None)
y_pred = next((globals()[k] for k in pred_candidates if k in globals()), None)
if (y_true is None or y_pred is None) and "best_trial" in globals() and best_trial.intermediate_values:
    steps = sorted(best_trial.intermediate_values.keys())
    vals = np.asarray([best_trial.intermediate_values[s] for s in steps], dtype=float)

    if y_true is None:
        y_true = vals
    if y_pred is None:
        # proxy prediction curve (running best)
        y_pred = np.minimum.accumulate(vals)

if y_true is None or y_pred is None:
    raise ValueError(
        "Could not find true/prediction arrays. Define variables like "
        "`y_true` and `y_pred` first, or ensure `best_trial.intermediate_values` exists."
    )

y_true = np.asarray(y_true).reshape(-1)
y_pred = np.asarray(y_pred).reshape(-1)

if y_true.shape[0] != y_pred.shape[0]:
    raise ValueError(f"Length mismatch: y_true={len(y_true)}, y_pred={len(y_pred)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sequence comparison
axes[0].plot(y_true, label="True", linewidth=2)
axes[0].plot(y_pred, label="Prediction", linewidth=2, alpha=0.8)
axes[0].set_title("True vs Prediction (Sequence)")
axes[0].set_xlabel("Sample")
axes[0].set_ylabel("Value")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Parity plot
axes[1].scatter(y_true, y_pred, alpha=0.7)
min_v = min(y_true.min(), y_pred.min())
max_v = max(y_true.max(), y_pred.max())
axes[1].plot([min_v, max_v], [min_v, max_v], "r--", linewidth=1.5)
axes[1].set_title("Parity Plot")
axes[1].set_xlabel("True")
axes[1].set_ylabel("Prediction")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

fig.savefig("true_vs_prediction_plots.png", dpi=300, bbox_inches="tight")
